In [1]:
# IMPORTS

import pandas as pd
import numpy as np

In [2]:
# BUILD SIGNAL PANEL FUNCTION

def build_signal_panel(data_dir="data", debug=True):
    import os
    import pandas as pd

    # --- Find all signal files ---
    signal_files = sorted([
        f for f in os.listdir(data_dir)
        if f.endswith("_signal.parquet")
    ])

    if len(signal_files) == 0:
        raise ValueError(f"No *_signal.parquet files found in {data_dir}")

    if debug:
        print("signal files found:")
        for f in signal_files:
            print(" -", f)

    # --- Load individual signal files ---
    panel_parts = []

    for file in signal_files:
        signal_name = file.replace("_signal.parquet", "")
        path = os.path.join(data_dir, file)

        sig = pd.read_parquet(path)

        if debug:
            print("\n" + "=" * 80)
            print("file:", file)
            print("signal name:", signal_name)
            print("shape:", sig.shape)
            print("index min:", sig.index.min() if len(sig.index) else None)
            print("index max:", sig.index.max() if len(sig.index) else None)
            print("columns:", sig.columns.tolist())
            print("non-null values:", sig.notna().sum().sum())

        # --- Skip unusable signals ---
        if sig.empty:
            print(f"SKIPPING {signal_name}: empty DataFrame")
            continue

        if sig.notna().sum().sum() == 0:
            print(f"SKIPPING {signal_name}: all values are NaN")
            continue

        # --- Clean index ---
        sig = sig[~sig.index.duplicated(keep="last")].sort_index()

        # --- Ensure float dtype (avoid downstream issues) ---
        sig = sig.astype(float)

        # --- Add signal level to columns ---
        sig.columns = pd.MultiIndex.from_product([[signal_name], sig.columns])
        panel_parts.append(sig)

    if len(panel_parts) == 0:
        raise ValueError("No usable signal files loaded")

    # --- Combine into one signal panel using OUTER join ---
    # NOTE: outer is correct here for standalone sleeve evaluation
    signal_panel = pd.concat(panel_parts, axis=1, join="outer")

    # --- Sort for consistency ---
    signal_panel = signal_panel.sort_index().sort_index(axis=1)
    signal_panel.columns.names = ["signal", "asset"]

    # --- Basic sanity: drop rows where everything is NaN ---
    signal_panel = signal_panel.dropna(how="all")

    if debug:
        print("\n" + "=" * 80)
        print("FINAL SIGNAL PANEL")
        print("shape:", signal_panel.shape)
        print("index min:", signal_panel.index.min())
        print("index max:", signal_panel.index.max())
        print("signals:", signal_panel.columns.get_level_values("signal").unique().tolist())
        print("assets:", signal_panel.columns.get_level_values("asset").unique().tolist())
        print("non-null values:", signal_panel.notna().sum().sum())

    return signal_panel

In [3]:
# BUILD SIGNAL PANEL
signal_panel = build_signal_panel()

# --- Set column level names safely ---
if isinstance(signal_panel.columns, pd.MultiIndex):
    signal_panel.columns.names = ["signal", "asset"]
else:
    raise ValueError("signal_panel columns are not MultiIndex")

# LOAD RETURNS
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# BASIC DIAGNOSTICS
print("signal_panel shape:", signal_panel.shape)
print("returns_df shape:", returns_df.shape)

# --- Get signal names safely ---
signal_names = (
    signal_panel.columns.get_level_values("signal").unique().tolist()
)
print("signals:", signal_names)

signal files found:
 - carry_signal.parquet
 - mean_reversion_signal.parquet
 - trend_cs_signal.parquet
 - trend_usd_factor_signal.parquet

file: carry_signal.parquet
signal name: carry
shape: (5196, 7)
index min: 2006-05-16 00:00:00
index max: 2026-05-05 00:00:00
columns: ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'JPY', 'NZD']
non-null values: 36372

file: mean_reversion_signal.parquet
signal name: mean_reversion
shape: (4532, 7)
index min: 2007-02-19 00:00:00
index max: 2026-05-05 00:00:00
columns: ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'JPY', 'NZD']
non-null values: 31061

file: trend_cs_signal.parquet
signal name: trend_cs
shape: (4523, 7)
index min: 2007-05-31 00:00:00
index max: 2026-05-05 00:00:00
columns: ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'JPY', 'NZD']
non-null values: 31164

file: trend_usd_factor_signal.parquet
signal name: trend_usd_factor
shape: (5196, 7)
index min: 2006-05-16 00:00:00
index max: 2026-05-05 00:00:00
columns: ['AUD', 'CAD', 'CHF', 'EUR', 'GBP', 'JPY', 'NZD']
non-nu

In [4]:
# STANDALONE SLEEVE PERFORMANCE

def performance_stats(pnl):
    pnl = pnl.dropna()

    ann = 252

    if len(pnl) == 0 or pnl.std() == 0:
        return {"Sharpe": np.nan, "MaxDD": np.nan}

    sharpe = pnl.mean() / pnl.std() * np.sqrt(ann)

    cum = (1 + pnl).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()

    return {
        "Sharpe": sharpe,
        "MaxDD": max_dd,
    }


def build_equal_weight_sleeve(
    signal_panel,
    returns_df,
    signal_name,
    alpha=0.1,
    min_cross_section=5,
    debug=False
):
    # --- Extract signal sleeve ---
    sig = signal_panel.xs(signal_name, axis=1, level="signal").copy()

    # --- Align index and assets ---
    common_idx = sig.index.intersection(returns_df.index)
    common_cols = sig.columns.intersection(returns_df.columns)

    if len(common_idx) == 0:
        raise ValueError(f"{signal_name}: no overlapping dates")

    if len(common_cols) == 0:
        raise ValueError(f"{signal_name}: no overlapping assets")

    sig = sig.loc[common_idx, common_cols]
    rets = returns_df.loc[common_idx, common_cols]

    # --- Require enough usable assets ---
    valid_rows = (
        (sig.notna().sum(axis=1) >= min_cross_section) &
        (rets.notna().sum(axis=1) >= min_cross_section)
    )

    sig = sig.loc[valid_rows]
    rets = rets.loc[valid_rows]

    if sig.empty:
        raise ValueError(f"{signal_name}: no valid rows after coverage filter")

    # --- Use signal directly ---
    weights = sig.copy()

    # --- Normalize to gross = 1 ---
    gross = weights.abs().sum(axis=1)
    weights = weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

    # --- Smooth weights ---
    weights = weights.ewm(alpha=alpha, adjust=False).mean()

    # --- Lag execution ---
    pnl = (weights.shift(1) * rets).sum(axis=1)

    if debug:
        print("\n" + "=" * 80)
        print("signal:", signal_name)
        print("signal shape:", sig.shape)
        print("returns shape:", rets.shape)
        print("common dates:", len(common_idx))
        print("common assets:", common_cols.tolist())
        print("signal non-null:", sig.notna().sum().sum())
        print("gross mean before norm:", gross.mean())
        print("weights gross mean:", weights.abs().sum(axis=1).mean())
        print("pnl count:", pnl.count())
        print("pnl std:", pnl.std())

    return weights, pnl


def compute_turnover(weights):
    return 0.5 * weights.diff().abs().sum(axis=1)


def evaluate_standalone_sleeves(
    signal_panel,
    returns_df,
    cost_per_turnover=0.0005,
    alpha=1.0,
    min_cross_section=5,
    debug=False
):
    # --- Validate panel ---
    if not isinstance(signal_panel.columns, pd.MultiIndex):
        raise ValueError("signal_panel columns must be MultiIndex")

    signal_panel.columns.names = ["signal", "asset"]

    rows = []
    sleeve_weights = {}
    sleeve_pnls = {}
    sleeve_net_pnls = {}
    sleeve_turnover = {}

    signal_names = signal_panel.columns.get_level_values("signal").unique().tolist()

    for signal_name in signal_names:
        weights, pnl = build_equal_weight_sleeve(
            signal_panel=signal_panel,
            returns_df=returns_df,
            signal_name=signal_name,
            alpha=alpha,
            min_cross_section=min_cross_section,
            debug=debug
        )

        turnover = compute_turnover(weights)
        net_pnl = pnl - turnover.fillna(0) * cost_per_turnover

        gross_stats = performance_stats(pnl)
        net_stats = performance_stats(net_pnl)

        sleeve_weights[signal_name] = weights
        sleeve_pnls[signal_name] = pnl
        sleeve_net_pnls[signal_name] = net_pnl
        sleeve_turnover[signal_name] = turnover

        rows.append({
            "Sleeve": signal_name,
            "NetSharpe": net_stats["Sharpe"],
            "GrossSharpe": gross_stats["Sharpe"],
            "MaxDD": net_stats["MaxDD"],
            "Turnover": turnover.mean() * 252,
        })

    sleeve_stats = (
        pd.DataFrame(rows)
        .set_index("Sleeve")
        .sort_values("NetSharpe", ascending=False)
    )

    return sleeve_stats, sleeve_weights, sleeve_pnls, sleeve_net_pnls, sleeve_turnover


# RUN
sleeve_stats, sleeve_weights, sleeve_pnls, sleeve_net_pnls, sleeve_turnover = evaluate_standalone_sleeves(
    signal_panel=signal_panel,
    returns_df=returns_df,
    cost_per_turnover=0.0005,
    alpha=.1, # signal smoothing 
    min_cross_section=5,
    debug=False
)

sleeve_stats

,NetSharpe,GrossSharpe,MaxDD,Turnover
Sleeve,,,,
carry,0.293623,0.295742,-0.215946,0.271719
mean_reversion,0.127907,0.147332,-0.214729,2.617154
trend_cs,0.124476,0.147399,-0.164288,2.429168
trend_usd_factor,0.056721,0.297307,-0.062788,8.029470


In [5]:
# LOAD REGIME STATES

states_df = pd.read_parquet("data/states_df.parquet")

print("states_df shape:", states_df.shape)
print("states:", states_df.columns.tolist())

states_df shape: (9231, 4)
states: ['equity_state', 'commodity_state', 'vol_state', 'funding_state']


In [6]:
# SLEEVE PERFORMANCE BY NUMERIC STATE REGIME

def performance_stats_full(pnl):
    pnl = pnl.dropna()
    ann = 252

    if len(pnl) == 0:
        return {
            "PnL": np.nan,
            "Sharpe": np.nan,
            "Vol": np.nan,
            "MaxDD": np.nan,
            "Skew": np.nan,
        }

    vol_daily = pnl.std()
    sharpe = pnl.mean() / vol_daily * np.sqrt(ann) if vol_daily != 0 else np.nan
    vol = vol_daily * np.sqrt(ann)
    total_return = (1 + pnl).prod() - 1
    cum = (1 + pnl).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    skew = pnl.skew()

    return {
        "PnL": total_return,
        "Sharpe": sharpe,
        "Vol": vol,
        "MaxDD": max_dd,
        "Skew": skew,
    }


def classify_state_regime(x, threshold=1.0):
    if pd.isna(x):
        return np.nan
    elif x >= threshold:
        return "high"
    elif x <= -threshold:
        return "low"
    else:
        return "neutral"


def sleeve_stats_by_numeric_state_regime(
    sleeve_pnls,
    states_path="data/states_df.parquet",
    threshold=1.0,
    min_obs=126
):
    states_df = pd.read_parquet(states_path)

    rows = []

    for sleeve_name, pnl in sleeve_pnls.items():
        pnl = pnl.dropna()

        common_idx = pnl.index.intersection(states_df.index)
        pnl = pnl.loc[common_idx]
        states = states_df.loc[common_idx]

        for state_name in states.columns:
            regime_series = states[state_name].apply(
                lambda x: classify_state_regime(x, threshold=threshold)
            )

            for regime in ["low", "neutral", "high"]:
                regime_idx = regime_series[regime_series == regime].index
                regime_pnl = pnl.loc[pnl.index.intersection(regime_idx)]

                if len(regime_pnl) < min_obs:
                    continue

                stats = performance_stats_full(regime_pnl)

                rows.append({
                    "Sleeve": sleeve_name,
                    "State": state_name,
                    "Regime": regime,
                    "Obs": len(regime_pnl),
                    "AvgStateValue": states.loc[regime_pnl.index, state_name].mean(),
                    "PnL": stats["PnL"],
                    "Sharpe": stats["Sharpe"],
                    "Vol": stats["Vol"],
                    "MaxDD": stats["MaxDD"],
                    "Skew": stats["Skew"],
                })

    return pd.DataFrame(rows).sort_values(
        ["State", "Sleeve", "Regime"]
    ).reset_index(drop=True)


# RUN REGIME DIAGNOSTIC
sleeve_regime_stats = sleeve_stats_by_numeric_state_regime(
    sleeve_pnls=sleeve_pnls,
    states_path="data/states_df.parquet",
    threshold=1.0,
    min_obs=126
)

# PIVOT SHARPE FOR VISUAL READ
pivot = (
    sleeve_regime_stats
    .pivot_table(
        index=["State", "Sleeve"],
        columns="Regime",
        values="Sharpe"
    )
    .reindex(columns=["low", "neutral", "high"])
)

pivot

Regime                                 low   neutral      high
State           Sleeve                                        
commodity_state carry            -0.629668  0.102248  1.432159
                mean_reversion    0.282838  0.282786  0.055984
                trend_cs          0.567351 -0.499400  0.182658
                trend_usd_factor  0.407077  0.371080  0.363315
equity_state    carry            -1.669136  0.146675  1.399002
                mean_reversion    0.196775 -0.162173  0.923835
                trend_cs          0.680978 -0.106464 -0.613104
                trend_usd_factor  0.298023  0.551336  0.704479
funding_state   carry             2.266077  0.411012 -1.275441
                mean_reversion    0.479104 -0.032180  1.227085
                trend_cs         -0.975066  0.205089  0.252361
                trend_usd_factor  0.339361  0.273605  0.456281
vol_state       carry             1.137922  0.100226 -0.353170
                mean_reversion   -0.096171  0.490888 -0.039217
                trend_cs          0.301768  0.028123  0.199037
                trend_usd_factor  1.162240 -0.194355  0.305993

In [7]:
# BUILD STATE-CONDITIONAL SLEEVE SCALARS (EXPANDING WINDOW)
def build_state_conditional_sleeve_scalars_expanding(
    sleeve_pnls,
    states_df,
    lookback=504,
    min_obs=126,
    pnl_window=21,
    strength=0.25,
    max_scalar=1.50,
    min_scalar=0.50,
    smooth_window=21,
    persistence_alpha=0.05,
    convex_power=1.5,
    convex_cap=3.0
):

    # --- NORMALIZE STATES (TIME-SERIES STANDARDIZATION)
    states_df = states_df.copy()
    states_df = (states_df - states_df.mean()) / states_df.std()
    states_df = states_df.replace([np.inf, -np.inf], np.nan)

    sleeve_names = list(sleeve_pnls.keys())
    scalar_df = pd.DataFrame(index=states_df.index, columns=sleeve_names, dtype=float)
    raw_score_df = pd.DataFrame(index=states_df.index, columns=sleeve_names, dtype=float)

    dates = states_df.index

    # --- Build rolling sleeve PnL response ---
    sleeve_response = {
        sleeve_name: pnl.rolling(pnl_window).sum()
        for sleeve_name, pnl in sleeve_pnls.items()
    }

    for t in range(lookback, len(dates)):
        current_date = dates[t]
        window_idx = dates[t - lookback:t]

        scalar_inputs = {s: np.nan for s in sleeve_names}

        for sleeve_name, pnl_response in sleeve_response.items():
            pnl_window_series = pnl_response.loc[pnl_response.index.intersection(window_idx)]

            if len(pnl_window_series.dropna()) < min_obs:
                continue

            state_score = 0.0

            for state_name in states_df.columns:
                state_window = states_df[state_name].loc[
                    states_df.index.intersection(pnl_window_series.index)
                ]

                common_idx = pnl_window_series.index.intersection(state_window.index)

                x = state_window.loc[common_idx]
                y = pnl_window_series.loc[common_idx]

                valid = x.notna() & y.notna()
                x = x.loc[valid]
                y = y.loc[valid]

                if len(x) < min_obs:
                    continue

                if x.std() == 0 or y.std() == 0:
                    continue

                # --- Quantile edge
                high = y[x > x.quantile(0.67)].mean()
                low  = y[x < x.quantile(0.33)].mean()

                if pd.isna(high) or pd.isna(low):
                    continue

                raw_edge = (high - low)

                # --- Shrinkage (keep this)
                shrink = min(1.0, len(x) / (min_obs * 2))
                state_edge = raw_edge * shrink

                # --- Aggregate
                state_score += state_edge * states_df.loc[current_date, state_name]

            scalar_inputs[sleeve_name] = state_score

        # --- Build signal (no normalization)
        state_score_signal = pd.Series(scalar_inputs, dtype=float)

        # clean only
        state_score_signal = state_score_signal.replace([np.inf, -np.inf], np.nan)

        # --- Convex transform
        convex_signal = np.sign(state_score_signal) * (np.abs(state_score_signal) ** convex_power)
        convex_signal = convex_signal.clip(-convex_cap, convex_cap)

        scalar = np.exp(strength * convex_signal)
        scalar = scalar.clip(lower=min_scalar, upper=max_scalar)

        raw_score_df.loc[current_date, state_score_signal.index] = state_score_signal
        scalar_df.loc[current_date, scalar.index] = scalar

    # --- Smooth (keep light)
    raw_score_df = raw_score_df.rolling(smooth_window, min_periods=1).mean()

    # --- Persistence filter
    scalar_df = scalar_df.ewm(alpha=persistence_alpha, adjust=False).mean()

    # --- Lag for execution
    raw_score_df = raw_score_df.shift(1)
    scalar_df = scalar_df.shift(1)

    return scalar_df, raw_score_df

In [9]:
# sleeve scalar build
sleeve_scalars_strict, sleeve_state_scores_strict = build_state_conditional_sleeve_scalars_expanding(
    sleeve_pnls=sleeve_pnls,
    states_df=states_df,
    lookback=756,            # history window for estimating state→pnl relationship
    min_obs=252,             # minimum data required to trust estimate
    pnl_window=21,           # horizon over which sleeve performance is measured
    strength=0.10,           # sensitivity of scalar to signal (tilt aggressiveness)
    max_scalar=1.20,         # upper cap on overweighting a sleeve
    min_scalar=0.80,         # lower cap on underweighting a sleeve
    smooth_window=21,        # rolling smoothing of raw signal
    persistence_alpha=0.1,   # EWMA smoothing of scalar 
    convex_power=1.0,        # nonlinearity applied to signal magnitude
    convex_cap=2.0           # cap on signal before exponential mapping
)

In [10]:
# EQUAL WEIGHT SLEEVE COMBO PORTFOLIO

def build_equal_sleeve_combo_pnl(
    sleeve_pnls,
    sleeve_turnover,
    cost_per_turnover=0.0005
):
    # --- Combine sleeve PnLs ---
    sleeve_pnls_df = pd.DataFrame(sleeve_pnls).dropna(how="all")
    sleeve_pnls_df = sleeve_pnls_df.dropna()

    # --- Equal weight sleeves ---
    combo_pnl = sleeve_pnls_df.mean(axis=1)

    # --- Combine turnover ---
    sleeve_turnover_df = pd.DataFrame(sleeve_turnover).loc[sleeve_pnls_df.index]
    combo_turnover = sleeve_turnover_df.mean(axis=1)

    # --- Costs ---
    combo_net_pnl = combo_pnl - combo_turnover.fillna(0) * cost_per_turnover

    return combo_pnl, combo_net_pnl, combo_turnover, sleeve_pnls_df


combo_pnl, combo_net_pnl, combo_turnover, sleeve_pnls_df = build_equal_sleeve_combo_pnl(
    sleeve_pnls=sleeve_pnls,
    sleeve_turnover=sleeve_turnover,
    cost_per_turnover=0.0005
)

combo_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(combo_pnl)["Sharpe"],
    "NetSharpe": performance_stats(combo_net_pnl)["Sharpe"],
    "MaxDD": performance_stats(combo_net_pnl)["MaxDD"],
    "Turnover": combo_turnover.mean() * 252
}, index=["equal_sleeve_combo"])

In [11]:
# INVERSE VOL SLEEVE COMBO PORTFOLIO

def build_inverse_vol_sleeve_combo_pnl(
    sleeve_pnls,
    sleeve_turnover,
    vol_window=60,
    vol_floor=0.01,
    cost_per_turnover=0.0005
):
    # --- Combine sleeve PnLs ---
    sleeve_pnls_df = pd.DataFrame(sleeve_pnls).dropna(how="all")
    sleeve_pnls_df = sleeve_pnls_df.dropna()

    # --- Estimate lagged sleeve vol ---
    sleeve_vol = sleeve_pnls_df.rolling(vol_window).std() * np.sqrt(252)
    sleeve_vol = sleeve_vol.shift(1)

    # --- Inverse vol sleeve allocation ---
    inv_vol = 1 / sleeve_vol.clip(lower=vol_floor)
    sleeve_alloc = inv_vol.div(inv_vol.sum(axis=1), axis=0).fillna(0)

    # --- Align allocations ---
    sleeve_alloc = sleeve_alloc.loc[sleeve_pnls_df.index]

    # --- Combine sleeve PnLs using inverse vol weights ---
    combo_pnl = (sleeve_pnls_df * sleeve_alloc).sum(axis=1)

    # --- Combine turnover using same sleeve allocations ---
    sleeve_turnover_df = pd.DataFrame(sleeve_turnover).loc[sleeve_pnls_df.index]
    combo_turnover = (sleeve_turnover_df * sleeve_alloc).sum(axis=1)

    # --- Costs ---
    combo_net_pnl = combo_pnl - combo_turnover.fillna(0) * cost_per_turnover

    return combo_pnl, combo_net_pnl, combo_turnover, sleeve_alloc, sleeve_pnls_df


ivol_combo_pnl, ivol_combo_net_pnl, ivol_combo_turnover, ivol_sleeve_alloc, sleeve_pnls_df = (
    build_inverse_vol_sleeve_combo_pnl(
        sleeve_pnls=sleeve_pnls,
        sleeve_turnover=sleeve_turnover,
        vol_window=60,
        vol_floor=0.01,
        cost_per_turnover=0.0005
    )
)

ivol_combo_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(ivol_combo_pnl)["Sharpe"],
    "NetSharpe": performance_stats(ivol_combo_net_pnl)["Sharpe"],
    "MaxDD": performance_stats(ivol_combo_net_pnl)["MaxDD"],
    "Turnover": ivol_combo_turnover.mean() * 252
}, index=["inverse_vol_sleeve_combo"])


In [12]:
# SCALAR WEIGHTED SLEEVE COMBO PORTFOLIO

def build_scalar_weighted_sleeve_combo_pnl(
    sleeve_pnls,
    sleeve_turnover,
    sleeve_scalars,
    cost_per_turnover=0.0005,
    alloc_alpha=0.1
):
    # --- Combine sleeve PnLs ---
    sleeve_pnls_df = pd.DataFrame(sleeve_pnls).dropna(how="all")
    sleeve_pnls_df = sleeve_pnls_df.dropna()

    # --- Align scalars ---
    common_idx = sleeve_pnls_df.index.intersection(sleeve_scalars.index)
    common_cols = sleeve_pnls_df.columns.intersection(sleeve_scalars.columns)

    sleeve_pnls_df = sleeve_pnls_df.loc[common_idx, common_cols]
    sleeve_alloc = sleeve_scalars.loc[common_idx, common_cols].copy()

    # --- Normalize scalars into sleeve weights ---
    sleeve_alloc = sleeve_alloc.div(
        sleeve_alloc.sum(axis=1).replace(0, np.nan),
        axis=0
    ).fillna(0)

    # --- Smooth sleeve allocations ---
    sleeve_alloc = sleeve_alloc.ewm(alpha=alloc_alpha, adjust=False).mean()

    # --- Combine sleeve PnLs ---
    combo_pnl = (sleeve_pnls_df * sleeve_alloc).sum(axis=1)

    # --- Combine turnover using same sleeve allocations ---
    sleeve_turnover_df = pd.DataFrame(sleeve_turnover).loc[sleeve_pnls_df.index, common_cols]
    combo_turnover = (sleeve_turnover_df * sleeve_alloc).sum(axis=1)

    # --- Costs ---
    combo_net_pnl = combo_pnl - combo_turnover.fillna(0) * cost_per_turnover

    return combo_pnl, combo_net_pnl, combo_turnover, sleeve_alloc, sleeve_pnls_df


scalar_combo_pnl, scalar_combo_net_pnl, scalar_turnover, scalar_sleeve_alloc, scalar_sleeve_pnls_df = (
    build_scalar_weighted_sleeve_combo_pnl(
        sleeve_pnls=sleeve_pnls,
        sleeve_turnover=sleeve_turnover,
        sleeve_scalars=sleeve_scalars_strict,
        cost_per_turnover=0.0005,
        alloc_alpha=0.1
    )
)

scalar_combo_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(scalar_combo_pnl)["Sharpe"],
    "NetSharpe": performance_stats(scalar_combo_net_pnl)["Sharpe"],
    "MaxDD": performance_stats(scalar_combo_net_pnl)["MaxDD"],
    "Turnover": scalar_turnover.mean() * 252
}, index=["scalar_weighted_sleeve_combo"])


In [13]:
# collecting portfolio diagnostics 

def collect_portfolio_diagnostics(
    portfolio_name,
    pnl,
    net_pnl,
    turnover,
    returns_df,
    weights=None,
    sleeve_alloc=None,
    sleeve_pnls_df=None,
    save_dir="data/diagnostics"
):
    import os
    import pandas as pd

    os.makedirs(save_dir, exist_ok=True)

    # --- Align core series ---
    common_idx = pnl.index.intersection(net_pnl.index).intersection(turnover.index)

    if weights is not None:
        common_idx = common_idx.intersection(weights.index)

    pnl = pnl.loc[common_idx]
    net_pnl = net_pnl.loc[common_idx]
    turnover = turnover.loc[common_idx]

    if weights is not None:
        weights = weights.loc[common_idx]

    if sleeve_alloc is not None:
        sleeve_alloc = sleeve_alloc.loc[
            sleeve_alloc.index.intersection(common_idx)
        ]

    if sleeve_pnls_df is not None:
        sleeve_pnls_df = sleeve_pnls_df.loc[
            sleeve_pnls_df.index.intersection(common_idx)
        ]

    aligned_returns = returns_df.loc[
        returns_df.index.intersection(common_idx)
    ]

    # --- Combine pnl ---
    pnl_df = pd.DataFrame({
        "gross": pnl,
        "net": net_pnl
    })

    # --- Stats ---
    gross_stats = performance_stats(pnl)
    net_stats = performance_stats(net_pnl)

    stats_df = pd.DataFrame({
        "GrossSharpe": [gross_stats["Sharpe"]],
        "NetSharpe": [net_stats["Sharpe"]],
        "MaxDD": [net_stats["MaxDD"]],
        "Turnover": [turnover.mean() * 252]
    }, index=[portfolio_name])

    # --- Save ---
    base = f"{save_dir}/{portfolio_name}"

    pnl_df.to_parquet(f"{base}_pnl.parquet")
    turnover.to_frame("turnover").to_parquet(f"{base}_turnover.parquet")
    stats_df.to_parquet(f"{base}_stats.parquet")
    aligned_returns.to_parquet(f"{base}_returns.parquet")

    if weights is not None:
        weights.to_parquet(f"{base}_weights.parquet")

    if sleeve_alloc is not None:
        sleeve_alloc.to_parquet(f"{base}_sleeve_alloc.parquet")

    if sleeve_pnls_df is not None:
        sleeve_pnls_df.to_parquet(f"{base}_sleeve_pnls.parquet")

    return {
        "pnl": pnl_df,
        "turnover": turnover,
        "stats": stats_df,
        "weights": weights,
        "sleeve_alloc": sleeve_alloc,
        "sleeve_pnls_df": sleeve_pnls_df
    }

In [16]:
# portfolio results 

portfolio_results = {
    "equal_sleeve_combo": {
        "pnl": combo_pnl,
        "net_pnl": combo_net_pnl,
        "turnover": combo_turnover,
        "weights": None,
        "sleeve_alloc": None,
        "sleeve_pnls_df": sleeve_pnls_df
    },

    "inverse_vol_sleeve_combo": {
        "pnl": ivol_combo_pnl,
        "net_pnl": ivol_combo_net_pnl,
        "turnover": ivol_combo_turnover,
        "weights": None,
        "sleeve_alloc": ivol_sleeve_alloc,
        "sleeve_pnls_df": sleeve_pnls_df
    },

    "scalar_weighted_sleeve_combo": {
        "pnl": scalar_combo_pnl,
        "net_pnl": scalar_combo_net_pnl,
        "turnover": scalar_turnover,
        "weights": None,
        "sleeve_alloc": scalar_sleeve_alloc,
        "sleeve_pnls_df": scalar_sleeve_pnls_df
    }
}

all_diags = {}
all_stats = []

for name, res in portfolio_results.items():
    diag = collect_portfolio_diagnostics(
        portfolio_name=name,
        pnl=res["pnl"],
        net_pnl=res["net_pnl"],
        turnover=res["turnover"],
        returns_df=returns_df,
        weights=res.get("weights"),
        sleeve_alloc=res.get("sleeve_alloc"),
        sleeve_pnls_df=res.get("sleeve_pnls_df"),
        save_dir="data/diagnostics"
    )

    all_diags[name] = diag
    all_stats.append(diag["stats"])

all_stats_df = pd.concat(all_stats)
all_stats_df.to_parquet("data/diagnostics/all_portfolio_stats.parquet")

all_stats_df

,GrossSharpe,NetSharpe,MaxDD,Turnover
equal_sleeve_combo,0.680876,0.592194,-0.045709,3.353380
inverse_vol_sleeve_combo,0.773134,0.595686,-0.028868,4.584806
scalar_weighted_sleeve_combo,0.788269,0.705360,-0.045705,3.351252


In [ ]:
from itertools import product
import pandas as pd

# --- PRECOMPUTE ROLLING PNL WINDOWS (SPEEDUP)
pnl_windows = {
    21: {k: v.rolling(21).sum() for k, v in sleeve_pnls.items()},
    63: {k: v.rolling(63).sum() for k, v in sleeve_pnls.items()},
}

results = []

grid = {
    "strength": [0.05, 0.10],
    "persistence_alpha": [0.05, 0.1],
    "convex_power": [1.0, 1.5],
    "smooth_window": [21],
    "pnl_window": [21],
    "max_scalar": [1.2, 1.5],
    "min_scalar": [0.5, 0.8],
}

keys = list(grid.keys())
combos = list(product(*grid.values()))
total = len(combos)

for i, values in enumerate(combos, 1):
    params = dict(zip(keys, values))

    print(f"{i}/{total} | {params}")

    # --- skip invalid scalar ranges
    if params["min_scalar"] >= 1.0 or params["max_scalar"] <= 1.0:
        continue

    try:
        # --- use cached pnl window
        sleeve_response = pnl_windows[params["pnl_window"]]

        # --- build scalars
        scalars, _ = build_state_conditional_sleeve_scalars_expanding(
            sleeve_pnls=sleeve_response,
            states_df=states_df,
            lookback=756,
            min_obs=252,
            pnl_window=params["pnl_window"],
            strength=params["strength"],
            max_scalar=params["max_scalar"],
            min_scalar=params["min_scalar"],
            smooth_window=params["smooth_window"],
            persistence_alpha=params["persistence_alpha"],
            convex_power=params["convex_power"],
            convex_cap=2.0
        )

        # --- build portfolio (SCALAR WEIGHTED)
        pnl, net_pnl, turnover, *_ = build_scalar_weighted_sleeve_combo_pnl(
            sleeve_pnls=sleeve_pnls,
            sleeve_turnover=sleeve_turnover,
            sleeve_scalars=scalars,
        )

        stats = performance_stats(net_pnl)

        sharpe = stats["Sharpe"]
        maxdd = stats["MaxDD"]
        to = turnover.mean() * 252

        # --- early kill bad configs
        if pd.isna(sharpe) or sharpe < 0:
            continue

        # --- composite score
        score = sharpe - 0.3 * to + 0.1 * maxdd

        results.append({
            **params,
            "Sharpe": sharpe,
            "Turnover": to,
            "MaxDD": maxdd,
            "Score": score
        })

    except Exception as e:
        print("skip:", e)
        continue

# --- RESULTS
results_df = pd.DataFrame(results).sort_values("Score", ascending=False)

results_df.head(20)

1/32 | {'strength': 0.05, 'persistence_alpha': 0.05, 'convex_power': 1.0, 'smooth_window': 21, 'pnl_window': 21, 'max_scalar': 1.2, 'min_scalar': 0.5}
skip: name 'build_inverse_vol_scalar_overlay_combo_pnl' is not defined
2/32 | {'strength': 0.05, 'persistence_alpha': 0.05, 'convex_power': 1.0, 'smooth_window': 21, 'pnl_window': 21, 'max_scalar': 1.2, 'min_scalar': 0.8}
skip: name 'build_inverse_vol_scalar_overlay_combo_pnl' is not defined
3/32 | {'strength': 0.05, 'persistence_alpha': 0.05, 'convex_power': 1.0, 'smooth_window': 21, 'pnl_window': 21, 'max_scalar': 1.5, 'min_scalar': 0.5}


KeyboardInterrupt: 